In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from tqdm import tqdm

%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', device)

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

In [ ]:
%%sh
# Download the data - you need to do this only once
wget --no-verbose --output-document=image.jpg https://github.com/chrirupp/cv_course/raw/main/data/image.jpg
wget --no-verbose --output-document=dog_cutout.png https://github.com/chrirupp/cv_course/raw/main/data/dog_cutout.png

# Lecture 13 - Tracking

Lucas-Kanade from scratch, the aperture problem, optical flow (Lucas-Kanade for every pixel, Horn-Schunck and RAFT, scored against the true flow), and point tracking over a whole video: frame-to-frame KLT versus CoTracker3, which tracks many points jointly and reasons about occlusions.

We use a **synthetic** clip (the same as in Lecture 12) so that we know the true motion of every pixel and the true trajectory of every point, including when it is hidden.

Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)). Cells that load large pretrained models are faster with a GPU runtime (*Edit > Notebook settings > T4 GPU*).

In [ ]:
# A synthetic video with exact ground truth: a camera pans and zooms over a photo (the background),
# a dog walks through the scene and passes behind a pole that stands in the scene.
# Because we render it ourselves, we know where every pixel goes (flow) and when it is hidden (occlusion).
background = cv2.cvtColor(cv2.imread('image.jpg'), cv2.COLOR_BGR2RGB).astype(np.float32)
cutout = cv2.imread('dog_cutout.png', cv2.IMREAD_UNCHANGED)
cutout = cv2.resize(cutout, (0, 0), fx=0.75, fy=0.75, interpolation=cv2.INTER_AREA)
dog_rgb = cv2.cvtColor(cutout[:, :, :3], cv2.COLOR_BGR2RGB).astype(np.float32)
dog_alpha = cutout[:, :, 3].astype(np.float32) / 255

H, W, T = 360, 480, 48
POLE_X, POLE_WIDTH = 640, 36            # the pole, in background (world) coordinates

def camera(t):
    """scale and offset: background point X appears at s * (X - o) in frame t"""
    return 0.5 * (1 + 0.003 * t), np.array([40 + 5.0 * t, 60 + 1.0 * t])

def dog_position(t):
    """top-left corner of the dog in frame t (image coordinates)"""
    return np.array([20 + 6.0 * t, 120 + 8 * np.sin(t / 5)])

def layers(t):
    """the masks of the dog and the pole in frame t (the pole is in front of the dog)"""
    s, o = camera(t)
    px, py = dog_position(t)
    shift = np.float32([[1, 0, px], [0, 1, py]])
    dog_mask = cv2.warpAffine(dog_alpha, shift, (W, H))
    pole_mask = np.zeros((H, W), np.float32)
    x0, x1 = s * (POLE_X - o[0]), s * (POLE_X + POLE_WIDTH - o[0])
    pole_mask[:, max(int(round(x0)), 0):max(int(round(x1)), 0)] = 1
    return dog_mask, pole_mask, shift

def render(t):
    s, o = camera(t)
    A = np.float32([[s, 0, -s * o[0]], [0, s, -s * o[1]]])
    frame = cv2.warpAffine(background, A, (W, H), flags=cv2.INTER_LINEAR)
    dog_mask, pole_mask, shift = layers(t)
    dog = cv2.warpAffine(dog_rgb, shift, (W, H))
    frame = frame * (1 - dog_mask[..., None]) + dog * dog_mask[..., None]
    frame = frame * (1 - pole_mask[..., None]) + np.array([70, 60, 55]) * pole_mask[..., None]
    return np.clip(frame, 0, 255).astype(np.uint8)

def ground_truth_tracks(points, t0=0):
    """positions (T, N, 2) and visibility (T, N) of points given in frame t0"""
    points = np.asarray(points, np.float32)
    dog_mask, pole_mask, _ = layers(t0)
    xi = np.clip(points[:, 0].round().astype(int), 0, W - 1)
    yi = np.clip(points[:, 1].round().astype(int), 0, H - 1)
    on_dog = dog_mask[yi, xi] > 0.5
    s0, o0 = camera(t0)
    world = points / s0 + o0                    # background points in world coordinates
    tracks, visible = np.zeros((T, len(points), 2), np.float32), np.zeros((T, len(points)), bool)
    for t in range(T):
        s, o = camera(t)
        tracks[t] = np.where(on_dog[:, None], points - dog_position(t0) + dog_position(t), s * (world - o))
        dog_mask, pole_mask, _ = layers(t)
        x, y = tracks[t, :, 0], tracks[t, :, 1]
        inside = (x >= 0) & (x <= W - 1) & (y >= 0) & (y <= H - 1)
        xi, yi = np.clip(x.round().astype(int), 0, W - 1), np.clip(y.round().astype(int), 0, H - 1)
        hidden = (pole_mask[yi, xi] > 0.5) | (~on_dog & (dog_mask[yi, xi] > 0.5))
        visible[t] = inside & ~hidden
    return tracks, visible

def ground_truth_flow(t):
    """dense optical flow from frame t to frame t+1 (H, W, 2)"""
    ys, xs = np.mgrid[0:H, 0:W].astype(np.float32)
    points = np.stack([xs.ravel(), ys.ravel()], axis=1)
    s0, o0 = camera(t)
    s1, o1 = camera(t + 1)
    flow = s1 * (points / s0 + o0 - o1) - points
    dog_mask, pole_mask, _ = layers(t)
    flow[dog_mask.ravel() > 0.5] = dog_position(t + 1) - dog_position(t)
    flow[pole_mask.ravel() > 0.5] = s1 * (points[pole_mask.ravel() > 0.5] / s0 + o0 - o1) - points[pole_mask.ravel() > 0.5]
    return flow.reshape(H, W, 2).astype(np.float32)

video = np.stack([render(t) for t in range(T)])
print('video:', video.shape)

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def animate(frames, interval=80):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    im = ax.imshow(frames[0])
    ax.axis('off')
    anim = FuncAnimation(fig, lambda t: [im.set_data(frames[t])], frames=len(frames), interval=interval)
    plt.close()
    return HTML(anim.to_jshtml())

animate(video)

## Lucas-Kanade for a translation

We want the displacement $\mathbf{d}$ that aligns a template $T$ (a patch of frame 0) with the next frame $I$:
$$E(\mathbf{d}) = \sum_{\mathbf{x}} \left[ I(\mathbf{x} + \mathbf{d}) - T(\mathbf{x}) \right]^2 .$$
Linearising $I$ around the current estimate gives the Gauss-Newton update
$$\Delta \mathbf{d} = - \Big(\sum \nabla I \nabla I^\top\Big)^{-1} \sum \nabla I \left[ I(\mathbf{x} + \mathbf{d}) - T(\mathbf{x}) \right],$$
which we iterate. The linearisation is only valid for small displacements.

In [ ]:
gray = np.stack([cv2.GaussianBlur(cv2.cvtColor(f, cv2.COLOR_RGB2GRAY).astype(np.float32), (0, 0), 1.0) for f in video])

def patch(image, center, size):
    return cv2.getRectSubPix(image, (size, size), (float(center[0]), float(center[1])))

def lucas_kanade(I0, I1, p, size=31, d=(0, 0), iterations=30):
    """estimate the translation of the patch around p from I0 to I1; returns all iterates"""
    Iy, Ix = np.gradient(I1)
    T = patch(I0, p, size)
    d = np.array(d, np.float64)
    history = [d.copy()]
    for _ in range(iterations):
        q = p + d
        error = patch(I1, q, size) - T
        gx, gy = patch(Ix, q, size).ravel(), patch(Iy, q, size).ravel()
        G = np.array([[gx @ gx, gx @ gy], [gx @ gy, gy @ gy]])   # structure tensor
        b = np.array([gx @ error.ravel(), gy @ error.ravel()])
        d = d - np.linalg.solve(G, b)
        history.append(d.copy())
    return np.array(history)

def lucas_kanade_pyramid(I0, I1, p, size=31, levels=3, iterations=30):
    """coarse-to-fine: estimate at low resolution, double, refine at the next level"""
    d = np.zeros(2)
    for level in reversed(range(levels)):
        s = 2 ** level
        small0 = cv2.resize(I0, (0, 0), fx=1/s, fy=1/s, interpolation=cv2.INTER_AREA)
        small1 = cv2.resize(I1, (0, 0), fx=1/s, fy=1/s, interpolation=cv2.INTER_AREA)
        d = lucas_kanade(small0, small1, p / s, size, d / s, iterations)[-1] * s
    return d

# a well-textured background point (the strongest corner in a region far from the dog)
min_eig = cv2.cornerMinEigenVal(gray[0], blockSize=15, ksize=3)
region = np.zeros_like(min_eig, bool)
region[100:300, 340:440] = True
y, x = np.unravel_index(np.argmax(np.where(region, min_eig, 0)), min_eig.shape)
p = np.array([x, y], np.float64)
def lucas_kanade_pyramid_history(I0, I1, p, size=31, levels=3, iterations=10):
    # coarse-to-fine, keeping every iterate (in full-resolution pixels) to compare with plain Lucas-Kanade
    d, history = np.zeros(2), []
    for level in reversed(range(levels)):
        s = 2 ** level
        small0 = cv2.resize(I0, (0, 0), fx=1/s, fy=1/s, interpolation=cv2.INTER_AREA)
        small1 = cv2.resize(I1, (0, 0), fx=1/s, fy=1/s, interpolation=cv2.INTER_AREA)
        h = lucas_kanade(small0, small1, p / s, size, d / s, iterations) * s
        history += list(h if not history else h[1:])
        d = h[-1]
    return np.array(history)

R = 45                                                   # half-size of the crop around the template
fig, axs = plt.subplots(1, 3, figsize=(24, 6.5), gridspec_kw={'width_ratios': [0.8, 1, 1]})
ax = axs[0]
ax.imshow(video[0][int(p[1]) - R:int(p[1]) + R + 1, int(p[0]) - R:int(p[0]) + R + 1])
ax.add_patch(plt.Rectangle((R - 15.5, R - 15.5), 31, 31, fill=False, color='red', linewidth=3))
for t, c in [(2, 'tab:blue'), (10, 'tab:purple')]:
    true = ground_truth_tracks([p])[0][t, 0] - p
    ax.add_patch(plt.Rectangle((R + true[0] - 15.5, R + true[1] - 15.5), 31, 31, fill=False, color=c, linewidth=3, ls='--'))
    ax.annotate('', xy=(R + true[0], R + true[1]), xytext=(R, R), arrowprops=dict(arrowstyle='->', lw=4, color=c),
                zorder=12 - t)
    ax.plot([], [], color=c, lw=3, label=f'to frame {t}: {np.linalg.norm(true):.1f} px')
ax.set_title('Template in frame 0 (red),\nwhere it moves (dashed)', fontsize=22)
ax.legend(fontsize=16, loc='lower right')
ax.axis('off')
for t, ax in zip([2, 10], axs[1:]):
    true = ground_truth_tracks([p])[0][t, 0] - p
    lk = np.linalg.norm(lucas_kanade(gray[0], gray[t], p) - true, axis=1)
    ax.plot(lk, 'o-', color='tab:blue', lw=2.5, label='Lucas-Kanade')
    if t == 10:
        c2f = np.linalg.norm(lucas_kanade_pyramid_history(gray[0], gray[t], p) - true, axis=1)
        ax.plot(c2f, 's-', color='tab:green', lw=2.5, label='coarse-to-fine: 1/4, 1/2, full size')
        for k in (10, 20):
            ax.axvline(k, color='tab:green', ls=':', lw=1.5)
    ax.axhline(0, color='black', lw=1)
    ax.set_xlabel('iteration', fontsize=18)
    ax.set_ylabel('distance to the true position (px)', fontsize=18)
    ax.set_title(f'Frame 0 → {t}: moved {np.linalg.norm(true):.1f} px', fontsize=22)
    ax.tick_params(labelsize=15)
    ax.legend(fontsize=16, loc='upper right')
    ax.grid(alpha=0.3)
    print(f'frame {t}: LK error after 1, 2, 3, 30 iterations:', np.round(lk[[1, 2, 3, 30]], 2))
    if t == 10:
        print('coarse-to-fine error after each level:', np.round(c2f[[10, 20, 30]], 2),
              'first iteration below 0.5 px: LK', np.argmax(lk < 0.5), 'c2f', np.argmax(c2f < 0.5))
plt.tight_layout()
savefig('tracking_lucas_kanade')
plt.show()

### Inside one Lucas-Kanade step

The update is easiest to see in 1D: the template is a short window of a row, and the row has moved by $d$ in the new frame. At a pixel where the image has slope $I'$, an intensity error $T - I$ means the image has to move by about $(T - I) / I'$: error divided by slope. Every pixel of the window gives such a guess; least squares combines them, weighted by $I'^2$ (steep pixels are trusted, flat ones ignored):
$$\delta = \frac{\sum I' \,(T - I)}{\sum I'^2}.$$
The slope is only a local approximation, so we move by $\delta$, recompute the error and repeat. In 2D, $\sum I'^2$ becomes the $2 \times 2$ matrix $G = \sum \nabla I \nabla I^\top$ and $\sum I'(T - I)$ the vector $b = \sum \nabla I\, (T - I)$: $\delta = G^{-1} b$.

In [ ]:
# Lucas-Kanade in 1D
# the image I is one row of frame 0; the template T is a short window of it, cut out where the content was
# before it moved by d: T(x) = I(x + d) for x in the window. LK slides the window until it fits.
row = gray[0][200, 100:260].astype(np.float64)
xs = np.arange(len(row), dtype=np.float64)
I1 = lambda x: np.interp(x, xs, row)
dI = lambda x: (I1(x + 0.5) - I1(x - 0.5))          # slope I'(x)
d_true = 4.0
win = np.arange(60, 81, dtype=np.float64)           # the template: 21 pixels, at its position in the previous frame
T1 = I1(win + d_true)

def lk_1d_step(d):
    e = T1 - I1(win + d)                             # error at every pixel of the window
    g = dI(win + d)
    return d + (g @ e) / (g @ g)                     # least squares: sum I'(T - I) / sum I'^2

estimates = [0.0]
for _ in range(5):
    estimates.append(lk_1d_step(estimates[-1]))
print('1D LK estimates of d =', d_true, ':', np.round(estimates, 3))

view = np.arange(45, 100, dtype=np.float64)
fig, axs = plt.subplots(1, 3, figsize=(24, 6.5))
ax = axs[0]
ax.plot(view, I1(view), color='tab:blue', lw=2, label='image I (new frame)')
ax.plot(win, T1, 'k-', lw=4, label='template T (old position)')
ax.axvspan(win[0], win[-1], color='0.85', zorder=0)
ax.set_title('Observed: template and image', fontsize=26)

ax = axs[1]
gm, em = dI(win), T1 - I1(win)
ok = (np.abs(gm) > 1.0) & (win > win[0] + 3) & (win < win[-1] - 3)                              # a pixel with a clear slope, where one step is nearly right
x0 = win[ok][np.argmin(np.abs(em[ok] / gm[ok] - d_true) - 0.05 * np.abs(em[ok]))]
e0, g0 = T1[win == x0][0] - I1(x0), dI(x0)
ax.plot(view, I1(view), color='tab:blue', lw=2, label='I')
ax.plot(win, T1, 'k-', lw=4, label='T')
tx = np.linspace(x0 - 3, x0 + e0 / g0 + 2, 10)
ax.plot(tx, I1(x0) + g0 * (tx - x0), '--', color='tab:orange', lw=2.5, label="tangent of I: slope I'")
ax.annotate('', xy=(x0, I1(x0) + e0), xytext=(x0, I1(x0)), arrowprops=dict(arrowstyle='<->', lw=2, color='tab:red'))
ax.text(x0 - 0.3, I1(x0) + e0 / 2, 'error\nT − I', color='tab:red', fontsize=20, va='center', ha='right', bbox=dict(fc='white', ec='none', alpha=0.85))
ax.annotate('', xy=(x0 + e0 / g0, I1(x0) + e0), xytext=(x0, I1(x0) + e0), arrowprops=dict(arrowstyle='->', lw=2.5, color='tab:green'))
ax.text(x0 - 0.3, I1(x0) + e0 - 1.5, "δ = (T − I) / I'", color='tab:green', fontsize=20, ha='right', va='top', bbox=dict(fc='white', ec='none', alpha=0.85))
ax.set_xlim(x0 - 8, x0 + 10)
ax.set_ylim(min(I1(x0), I1(x0) + e0) - 12, max(I1(x0), I1(x0) + e0) + 12)
ax.set_title('One pixel: error / slope = shift', fontsize=26)

ax = axs[2]
ax.plot(view, I1(view), color='tab:blue', lw=2, label='I')
colors = plt.cm.viridis(np.linspace(0, 0.85, 4))
for k, c in zip(range(4), colors):
    ax.plot(win + estimates[k], T1, color=c, lw=3.5, label=f'T moved by δ = {estimates[k]:.2f} (iteration {k})')
ax.set_title("All pixels: δ = Σ I'(T − I) / Σ I'², iterate", fontsize=26)
for ax in axs:
    ax.set_xlabel('x (pixel)', fontsize=16); ax.tick_params(labelsize=14); ax.grid(alpha=0.3)
    ax.legend(fontsize=16, loc='upper right')
for ax in axs[[0, 2]]:
    ax.set_xlim(view[0], view[-1]); ax.set_ylim(20, 99)
savefig('tracking_lk_1d')
plt.show()

In [ ]:
# the same in 2D: where the template comes from, then the patch at the current estimate and its error,
# iteration by iteration (frame 0 -> 4)
t_step = 4
history = lucas_kanade(gray[0], gray[t_step], p)
true_d = ground_truth_tracks([p])[0][t_step, 0] - p
show = [0, 1, 2, 3, len(history) - 1]
T_rgb = patch(video[0].astype(np.float32), p, 31)
r, R = T_rgb.shape[0] // 2, 60                      # template half-size, context half-size
context = lambda t: video[t][int(p[1]) - R:int(p[1]) + R + 1, int(p[0]) - R:int(p[0]) + R + 1]
box = lambda ax, d, color, ls='-': ax.add_patch(plt.Rectangle((R + d[0] - r - 0.5, R + d[1] - r - 0.5), 2 * r + 1, 2 * r + 1,
                                                              fill=False, ec=color, lw=5, ls=ls))
fig, axs = plt.subplots(2, len(show) + 2, figsize=(28, 12))
axs[0, 0].imshow(context(0)); box(axs[0, 0], (0, 0), 'red')
axs[0, 0].set_title('Frame 0\ncut out T', fontsize=34)
axs[1, 0].imshow(context(t_step)); box(axs[1, 0], (0, 0), 'red', '--'); box(axs[1, 0], true_d, 'lime')
axs[1, 0].set_title(f'Frame {t_step}: start (red),\ntrue (green)', fontsize=34)
axs[0, 1].imshow(T_rgb.astype(np.uint8))
axs[0, 1].set_title('Template T\n', fontsize=34)
for sp in axs[0, 1].spines.values():
    sp.set_color('red'); sp.set_linewidth(4)
axs[1, 1].axis('off')
for j, k in enumerate(show):
    d = history[k]
    P = patch(video[t_step].astype(np.float32), p + d, 31)
    err = np.abs(patch(gray[t_step], p + d, 31) - patch(gray[0], p, 31))
    axs[0, j + 2].imshow(P.astype(np.uint8))
    axs[0, j + 2].set_title(f'iteration {k}\n({d[0]:.1f}, {d[1]:.1f})', fontsize=34)
    axs[1, j + 2].imshow(err, cmap='gray_r', vmin=0, vmax=60)
    axs[1, j + 2].set_title(f'|T − I|\nmean {err.mean():.1f}', fontsize=34)
for ax in axs.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(w_pad=2.5)
savefig('tracking_lk_steps')
plt.show()

## The aperture problem

The update needs the inverse of the structure tensor $G = \sum \nabla I \nabla I^\top$. Its eigenvalues tell us whether a patch can be tracked: in a flat region both are small, on an edge only one is large (motion *along* the edge is invisible), at a corner both are large. Shi-Tomasi's "good features to track" are the points where the smaller eigenvalue is large.

In [ ]:
# the larger eigenvalue, to find a flat patch (both eigenvalues small)
max_eig = cv2.cornerEigenValsAndVecs(gray[0], blockSize=15, ksize=3)[:, :, 0]
inner = np.zeros_like(max_eig, bool)
inner[20:-20, 20:280] = True
y, x = np.unravel_index(np.argmin(np.where(inner, max_eig, np.inf)), max_eig.shape)
s, o = camera(0)
pole_left = s * (POLE_X - o[0])
examples = {'flat': np.array([x, y], np.float64), 'edge (of the pole)': np.array([pole_left, 60.0]), 'corner': p}

fig, axs = plt.subplots(1, 4, figsize=(24, 5), gridspec_kw={'width_ratios': [2, 1, 1, 1]})
axs[0].imshow(np.log(min_eig + 1e-3), cmap='magma')
axs[0].set_title('log of the smaller eigenvalue of the structure tensor')
for ax, (name, q) in zip(axs[1:], examples.items()):
    Iy, Ix = np.gradient(gray[0])
    gx, gy = patch(Ix, q, 31).ravel(), patch(Iy, q, 31).ravel()
    eig = np.linalg.eigvalsh(np.array([[gx @ gx, gx @ gy], [gx @ gy, gy @ gy]]))
    ax.imshow(patch(video[0].astype(np.float32), q, 31).astype(np.uint8))
    ax.set_title(f'{name}: eigenvalues {eig[0]:.0f}, {eig[1]:.0f}')
    axs[0].add_patch(plt.Rectangle(q - 15, 31, 31, fill=False, color='cyan', linewidth=2))
for ax in axs:
    ax.axis('off')
savefig('tracking_aperture')
plt.show()

## Optical flow: a displacement for every pixel

Brightness constancy linearised gives one equation per pixel, the *motion constraint* $I_x u + I_y v + I_t = 0$, for two unknowns $(u, v)$. On its own it fixes only the *normal flow*, the component along the gradient (the aperture problem again). Two classical ways to add the missing constraint:

* **Lucas-Kanade**: assume the flow is constant in a window around each pixel and solve the same $2 \times 2$ system as for template tracking, now at every pixel (sums over the window = a box filter).
* **Horn-Schunck**: ask for a *smooth* flow field, $\min \sum (I_x u + I_y v + I_t)^2 + \alpha^2 (|\nabla u|^2 + |\nabla v|^2)$, solved by iterating: each flow vector is pulled towards the average of its neighbours and corrected along the gradient.

Both linearise, so they only work for small motions: we run them **coarse-to-fine** on an image pyramid, warping the second frame with the current flow before estimating the rest. We compare them with RAFT (a learned model) using the **end-point error** (EPE), the mean distance between estimated and true flow vectors.

In [ ]:
from torchvision.models.optical_flow import raft_large, Raft_Large_Weights
from torchvision.utils import flow_to_image

def flow_rgb(flow, max_flow=12):
    # colour code as in Lecture 12: hue = direction, saturation = speed (saturated at max_flow px)
    flow = torch.from_numpy(np.ascontiguousarray(flow)).permute(2, 0, 1).clamp(-max_flow, max_flow)
    flow[0, 0, 0], flow[1, 0, 1] = max_flow, max_flow       # fix the colour scale across images
    return flow_to_image(flow).permute(1, 2, 0).numpy()

def epe(flow, gt):
    # end-point error: mean length of the difference vector (px)
    return np.linalg.norm(flow - gt, axis=2).mean()

def warp(image, flow):
    # sample the image at x + flow(x), so that the warped second frame lines up with the first
    h, w = image.shape
    ys, xs = np.mgrid[0:h, 0:w].astype(np.float32)
    return cv2.remap(image, xs + flow[..., 0], ys + flow[..., 1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

def derivatives(I0, I1):
    Iy, Ix = np.gradient(I1)
    return Ix, Iy, I1 - I0

def normal_flow(I0, I1):
    # the motion constraint alone: only the component along the gradient, -I_t grad(I) / |grad(I)|^2
    Ix, Iy, It = derivatives(I0, I1)
    g2 = Ix ** 2 + Iy ** 2 + 1e-2
    return np.stack([-It * Ix / g2, -It * Iy / g2], axis=2)

def lk_step(I0, I1, window=15, damping=10.0):
    # Lucas-Kanade at every pixel: G d = -b with sums over the window (a box filter); the damping keeps flat regions near 0
    Ix, Iy, It = derivatives(I0, I1)
    box = lambda a: cv2.boxFilter(a, -1, (window, window), normalize=False)
    gxx, gxy, gyy = box(Ix * Ix) + damping * window ** 2, box(Ix * Iy), box(Iy * Iy) + damping * window ** 2
    bx, by = box(Ix * It), box(Iy * It)
    det = gxx * gyy - gxy ** 2
    return np.stack([-(gyy * bx - gxy * by) / det, -(gxx * by - gxy * bx) / det], axis=2)

def hs_step(I0, I1, alpha=30.0, iterations=100):
    # Horn-Schunck: Jacobi iterations; each vector moves to the mean of its neighbours, corrected along the gradient
    Ix, Iy, It = derivatives(I0, I1)
    mean = np.array([[1, 2, 1], [2, 0, 2], [1, 2, 1]], np.float32) / 12
    u, v = np.zeros_like(I0), np.zeros_like(I0)
    for _ in range(iterations):
        ub, vb = cv2.filter2D(u, -1, mean), cv2.filter2D(v, -1, mean)
        r = (Ix * ub + Iy * vb + It) / (alpha ** 2 + Ix ** 2 + Iy ** 2)
        u, v = ub - Ix * r, vb - Iy * r
    return np.stack([u, v], axis=2)

def coarse_to_fine(step, I0, I1, levels=4, warps=3):
    # estimate on a pyramid: upsample (and double) the flow, warp frame 1 towards frame 0, estimate what is left
    pyr0, pyr1 = [I0], [I1]
    for _ in range(levels - 1):
        pyr0.append(cv2.pyrDown(pyr0[-1]))
        pyr1.append(cv2.pyrDown(pyr1[-1]))
    flow = np.zeros(pyr0[-1].shape + (2,), np.float32)
    for J0, J1 in zip(reversed(pyr0), reversed(pyr1)):
        if flow.shape[:2] != J0.shape:
            flow = 2 * cv2.resize(flow, (J0.shape[1], J0.shape[0]), interpolation=cv2.INTER_LINEAR)
        for _ in range(warps):
            flow = flow + step(J0, warp(J1, flow))
    return flow

raft = raft_large(weights=Raft_Large_Weights.DEFAULT).to(device).eval()

def raft_flow(frame0, frame1):
    # RAFT expects images in [-1, 1] with sides divisible by 8
    x0 = torch.from_numpy(frame0).permute(2, 0, 1)[None].float().div(127.5).sub(1).to(device)
    x1 = torch.from_numpy(frame1).permute(2, 0, 1)[None].float().div(127.5).sub(1).to(device)
    with torch.no_grad():
        return raft(x0, x1)[-1][0].permute(1, 2, 0).cpu().numpy()   # the last of 12 refinement iterations

In [ ]:
# frame 10 -> 11: the dog is in the open (it passes behind the pole around frame 26)
t = 10
gt = ground_truth_flow(t)
I0, I1 = gray[t], gray[t + 1]
flows = {'Lucas-Kanade': coarse_to_fine(lk_step, I0, I1),
         'Horn-Schunck': coarse_to_fine(hs_step, I0, I1),
         'RAFT': raft_flow(video[t], video[t + 1])}
for name, flow in flows.items():
    print(f'{name:14s} EPE {epe(flow, gt):.2f} px')

# the motion constraint alone vs. Horn-Schunck's smoothness
fig, axs = plt.subplots(1, 4, figsize=(24, 4.6))
panels = [(video[t], f'Frame {t}'), (flow_rgb(normal_flow(I0, I1)), f'Motion constraint only: EPE {epe(normal_flow(I0, I1), gt):.2f} px'),
          (flow_rgb(flows['Horn-Schunck']), f'Horn-Schunck: EPE {epe(flows["Horn-Schunck"], gt):.2f} px'), (flow_rgb(gt), 'Ground truth')]
def frame(ax):
    # a thin grey frame instead of axis('off'): flow images are white where nothing moves
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_color('0.6')

for ax, (image, title) in zip(axs, panels):
    ax.imshow(image)
    ax.set_title(title, fontsize=20)
    frame(ax)
savefig('flow_horn_schunck')
plt.show()

fig, axs = plt.subplots(1, 4, figsize=(24, 4.6))
axs[0].imshow(flow_rgb(gt))
axs[0].set_title('Ground truth', fontsize=22)
for ax, (name, flow) in zip(axs[1:], flows.items()):
    ax.imshow(flow_rgb(flow))
    ax.set_title(f'{name}: EPE {epe(flow, gt):.2f} px', fontsize=22)
for ax in axs:
    frame(ax)
savefig('flow_compare')
plt.show()

In [ ]:
# the colour code of the flow images (as in Lecture 12): every pixel of the disc gets the vector from the centre to it
R = 200
ys, xs = np.mgrid[-R:R + 1, -R:R + 1].astype(np.float32)
inside = xs ** 2 + ys ** 2 <= R ** 2
wheel = flow_rgb(np.stack([xs, ys], axis=2) * 12 / R * inside[..., None])
wheel[~inside] = 255
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(wheel, extent=(-R, R, R, -R))
for (dx, dy), name in [((1, 0), 'right'), ((-1, 0), 'left'), ((0, -1), 'up'), ((0, 1), 'down')]:
    ax.annotate('', xy=(dx * R * 0.8, dy * R * 0.8), xytext=(0, 0), arrowprops=dict(arrowstyle='->', lw=2.5, color='black'))
    ax.text(dx * R * (1.3 if dx else 1.13), dy * R * 1.13, name, ha='center', va='center', fontsize=20)
ax.set_xlim(-R * 1.6, R * 1.6); ax.set_ylim(R * 1.3, -R * 1.3)
ax.axis('off')
savefig('flow_wheel')
plt.show()

# mean EPE over the whole clip (RAFT is slow without a GPU)
errors = {name: [] for name in flows}
for t in tqdm(range(T - 1)):
    gt_t = ground_truth_flow(t)
    errors['Lucas-Kanade'].append(epe(coarse_to_fine(lk_step, gray[t], gray[t + 1]), gt_t))
    errors['Horn-Schunck'].append(epe(coarse_to_fine(hs_step, gray[t], gray[t + 1]), gt_t))
    errors['RAFT'].append(epe(raft_flow(video[t], video[t + 1]), gt_t))
plt.figure(figsize=(10, 4))
for name, e in errors.items():
    plt.plot(e, label=f'{name} (mean {np.mean(e):.2f} px)')
behind = [((layers(t)[0] > 0.5) & (layers(t)[1] > 0.5)).any() for t in range(T - 1)]
plt.fill_between(range(T - 1), 0, 1, where=behind, color='gray', alpha=0.2, transform=plt.gca().get_xaxis_transform(), label='dog behind the pole')
plt.xlabel('frame')
plt.ylabel('end-point error (px)')
plt.yscale('log')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Tracking points through the whole video

We pick points on the background and on the dog in frame 0 and follow them through all 48 frames.

* **KLT**: pyramidal Lucas-Kanade from frame to frame (OpenCV). Every step adds a little error (drift), and it has no notion of occlusion: when the dog walks behind the pole, its points get stuck on the pole or are lost.
* **CoTracker3**: a transformer that tracks all points *jointly* over a window of frames, using correlation features. It predicts both positions and visibility, and keeps tracking points while they are hidden.

In [ ]:
dog_mask, pole_mask, _ = layers(0)
corners = lambda mask, n: cv2.goodFeaturesToTrack(gray[0], n, 0.01, 8, mask=mask.astype(np.uint8)).reshape(-1, 2)
queries = np.concatenate([corners((dog_mask > 0.9) & (pole_mask < 0.5), 40),
                          corners((dog_mask < 0.1) & (pole_mask < 0.5), 100)])
on_dog = dog_mask[queries[:, 1].astype(int), queries[:, 0].astype(int)] > 0.5
true_tracks, true_visible = ground_truth_tracks(queries)
print(f'{len(queries)} query points, {on_dog.sum()} on the dog')

# KLT: chain pyramidal LK from frame to frame; a point is lost once its status is 0
klt = np.zeros_like(true_tracks)
klt_alive = np.zeros(true_visible.shape, bool)
klt[0], klt_alive[0] = queries, True
for t in range(1, T):
    nxt, status, _ = cv2.calcOpticalFlowPyrLK(video[t - 1], video[t], klt[t - 1].reshape(-1, 1, 2), None, winSize=(21, 21), maxLevel=3)
    klt[t] = nxt.reshape(-1, 2)
    klt_alive[t] = klt_alive[t - 1] & (status.ravel() == 1)
print(f'KLT lost {(~klt_alive[-1]).sum()} of {len(queries)} points')

In [ ]:
cotracker = torch.hub.load('facebookresearch/co-tracker', 'cotracker3_offline').to(device).eval()
video_tensor = torch.from_numpy(video).permute(0, 3, 1, 2)[None].float().to(device)        # (1, T, 3, H, W) in [0, 255]
query_tensor = torch.from_numpy(np.concatenate([np.zeros((len(queries), 1)), queries], axis=1)).float()[None].to(device)  # (t, x, y)
with torch.no_grad():
    tracks, visibility = cotracker(video_tensor, queries=query_tensor)
cotracker_tracks = tracks[0].cpu().numpy()
cotracker_visible = visibility[0].cpu().numpy()
print('occlusion prediction accuracy:', (cotracker_visible == true_visible).mean().round(3))

In [ ]:
def accuracy(tracks, valid, threshold=3):
    """per frame: fraction of all query points that are tracked to within `threshold` px of the truth
    (positions are scored also while a point is hidden - a tracker should know where it is)"""
    inside = (true_tracks[..., 0] >= 0) & (true_tracks[..., 0] < W)
    good = (np.linalg.norm(tracks - true_tracks, axis=2) < threshold) & valid
    return [(good[t][inside[t]]).mean() for t in range(T)], good

klt_acc, klt_good = accuracy(klt, klt_alive)
co_acc, co_good = accuracy(cotracker_tracks, np.ones_like(true_visible))
fig, axs = plt.subplots(1, 3, figsize=(24, 6), gridspec_kw={'width_ratios': [1.2, 1, 1]})
for name, acc, good in [('KLT', klt_acc, klt_good), ('CoTracker3', co_acc, co_good)]:
    axs[0].plot(acc, label=f'{name}: all points')
    axs[0].plot([good[t][on_dog].mean() for t in range(T)], '--', label=f'{name}: points on the dog')
hidden = ~true_visible[:, on_dog].all(axis=1)
axs[0].fill_between(range(T), 0, 1.02, where=hidden, color='gray', alpha=0.2, label='some dog points hidden')
axs[0].set_xlabel('frame')
axs[0].set_ylabel('fraction of points within 3 px')
axs[0].set_ylim(0, 1.02)
axs[0].legend(loc='lower left')
axs[0].grid(alpha=0.3)

t = 40
for ax, (name, tracks, valid, vis) in zip(axs[1:], [('KLT', klt, klt_alive, klt_alive), ('CoTracker3', cotracker_tracks, np.ones_like(true_visible), cotracker_visible)]):
    ax.imshow(video[t])
    for i in np.where(on_dog)[0]:
        if valid[t, i]:
            ax.plot(tracks[:t + 1, i, 0], tracks[:t + 1, i, 1], '-', color='yellow', linewidth=0.8, alpha=0.7)
    ok = np.linalg.norm(tracks[t] - true_tracks[t], axis=1) < 3
    sel = on_dog & valid[t]
    ax.scatter(*tracks[t, sel & ok].T, c='lime', s=25, label='within 3 px')
    ax.scatter(*tracks[t, sel & ~ok].T, c='red', s=25, label='wrong')
    if name == 'CoTracker3':
        ax.scatter(*tracks[t, sel & ~vis[t]].T, facecolors='none', edgecolors='white', s=80, label='predicted hidden')
    ax.set_title(f'{name}, frame {t}: points on the dog ({(~valid[t] & on_dog).sum()} lost)')
    ax.legend(loc='lower right')
    ax.axis('off')

if SAVE_FIGURES:   # the accuracy plot on its own, for the slide next to the GIF below
    bb = axs[0].get_tightbbox(fig.canvas.get_renderer()).transformed(fig.dpi_scale_trans.inverted())
    fig.savefig('figures/tracking_klt_accuracy.png', dpi=150, bbox_inches=bb.padded(0.1))
savefig('tracking_klt_vs_cotracker')
plt.show()

In [ ]:
# watch the dog points: KLT (left) and CoTracker3 (right); hollow circles = predicted hidden
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8), dpi=72)
ims = [ax.imshow(video[0]) for ax in axs]
dots = [axs[0].scatter([], [], c='yellow', s=12), axs[1].scatter([], [], c='yellow', s=12), axs[1].scatter([], [], facecolors='none', edgecolors='white', s=40)]
for ax, name in zip(axs, ['KLT', 'CoTracker3']):
    ax.set_title(name)
    ax.axis('off')
def update(t):
    for im in ims:
        im.set_data(video[t])
    dots[0].set_offsets(klt[t, on_dog & klt_alive[t]])
    dots[1].set_offsets(cotracker_tracks[t, on_dog & cotracker_visible[t]])
    dots[2].set_offsets(cotracker_tracks[t, on_dog & ~cotracker_visible[t]])
# the same as a GIF for the slides: dog points, green = within 3 px, red = wrong, white ring = predicted hidden
if SAVE_FIGURES:
    from PIL import Image
    def draw(t, tracks, valid, vis, name):
        im = video[t].copy()
        for i in np.where(on_dog & valid[t])[0]:
            cv2.polylines(im, [tracks[max(0, t - 8):t + 1, i].astype(np.int32)], False, (255, 255, 0), 1, cv2.LINE_AA)
            x, y = tracks[t, i].astype(int)
            if vis is not None and not vis[t, i]:
                cv2.circle(im, (x, y), 6, (255, 255, 255), 2, cv2.LINE_AA)
            else:
                ok = np.linalg.norm(tracks[t, i] - true_tracks[t, i]) < 3
                cv2.circle(im, (x, y), 4, (0, 230, 0) if ok else (255, 0, 0), -1, cv2.LINE_AA)
        cv2.putText(im, f'{name}  frame {t}', (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
        return im
    gap = np.full((video.shape[1], 10, 3), 255, np.uint8)
    gif = [Image.fromarray(np.concatenate([draw(t, klt, klt_alive, None, 'KLT'), gap,
                                           draw(t, cotracker_tracks, np.ones_like(true_visible), cotracker_visible, 'CoTracker3')], axis=1))
           for t in range(T)]
    gif = [g.resize((640, round(g.height * 640 / g.width)), Image.LANCZOS) for g in gif]
    palette = gif[0].quantize(128)                          # one palette, no dithering: a much smaller file
    gif = [g.quantize(palette=palette, dither=Image.Dither.NONE) for g in gif]
    gif[0].save('figures/tracking_klt_vs_cotracker.gif', save_all=True, append_images=gif[1:], duration=150, loop=0,
                optimize=True)
anim = FuncAnimation(fig, update, frames=T, interval=100)
plt.close()
HTML(anim.to_jshtml())